아날로그 시계 

In [ ]:
def solution(h1, m1, s1, h2, m2, s2):
    answer = -1
    # 초침: 1초 6도
    # 분침: 1초 0.1도
    # 시침: 1초 1/120도
    # 86400
    h=h2-h1
    m=m2-m1
    s=s2-s1
    
    time = h*3600 + m *60 + s
    print(time)
    hh = h1*3600*(1/120)
    mm = m1*60*(0.1)
    ss = s1*6
    print(hh,mm,ss)
    
    for t in range(5):
        hh += (1/120)
        mm += (0.1)
        ss += 6
        print(hh,mm,ss)
        
    # for t in range(time):
    #     hh = h1 + (1/120)*t
    #     mm = m1 + (0.1)*t
    #     ss = s1 + 6*t
    return answer

In [1]:
from fractions import Fraction
from math import floor
from decimal import Decimal, getcontext

getcontext().prec = 20  # 디스플레이용

def hms_to_seconds(hhmmss: str) -> int:
    h, m, s = map(int, hhmmss.split(':'))
    return h*3600 + m*60 + s

# 겹침 주기(초) - 정확한 유리수
T_sm = Fraction(3600, 59)   # 초 vs 분
T_sh = Fraction(43200, 719) # 초 vs 시

def count_hits_in_interval(A: int, B: int, T: Fraction) -> int:
    """구간 (A, B]에서 주기 T의 배수 시각 개수"""
    return floor(Fraction(B,1) / T) - floor(Fraction(A,1) / T)

def list_hits(A: int, B: int, T: Fraction):
    """구간 (A, B]의 겹침 시각(초)을 리스트로(Fraction)"""
    k_start = floor(Fraction(A,1) / T) + 1
    k_end   = floor(Fraction(B,1) / T)
    return [k*T for k in range(k_start, k_end+1)]

def fmt_time(t_frac: Fraction) -> str:
    # Fraction(초) -> "HH:MM:SS.sss"
    t = Decimal(t_frac.numerator) / Decimal(t_frac.denominator)
    total = int(t)  # 초
    ms = int((t - total) * Decimal(1000) + Decimal('0.5'))
    h = total // 3600
    m = (total % 3600) // 60
    s = (total % 60)
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

def debug_interval(start_str: str, end_str: str):
    A = hms_to_seconds(start_str)
    B = hms_to_seconds(end_str)
    print(f"\n[구간] ({start_str}, {end_str}]  → 초단위 ({A}, {B}]")

    # 각각의 겹침 시각
    sm_hits = list_hits(A, B, T_sm)
    sh_hits = list_hits(A, B, T_sh)

    # 중복 제거용(세트로 합치기)
    # 같은 시각(완전히 같은 Fraction)만 중복으로 간주
    unique_all = set(sm_hits) | set(sh_hits)

    print("\n- 초 vs 분 겹침 시각들:")
    for t in sm_hits[:10]:
        print("  ", fmt_time(t))
    if len(sm_hits) > 10:
        print(f"  ... 총 {len(sm_hits)}회")

    print("\n- 초 vs 시 겹침 시각들:")
    for t in sh_hits[:10]:
        print("  ", fmt_time(t))
    if len(sh_hits) > 10:
        print(f"  ... 총 {len(sh_hits)}회")

    # 중복(동시 겹침) 찾기
    duplicates = set(sm_hits) & set(sh_hits)

    print("\n- 동시 겹침(초-분 & 초-시 동시에 같은 순간):")
    if duplicates:
        for t in sorted(duplicates):
            print("  ", fmt_time(t))
    else:
        print("  (없음)")

    print("\n[카운트]")
    print(f"초-분: {len(sm_hits)}회")
    print(f"초-시: {len(sh_hits)}회")
    print(f"합계(중복 제거): {len(unique_all)}회")

# ===== 예시 디버깅 =====
debug_interval("00:00:00", "00:10:00")
debug_interval("01:23:45", "02:00:00")
# 12시간 전체: 동시 겹침(자정/정오) 확인
debug_interval("01:05:05", "01:05:06")



[구간] (00:00:00, 00:10:00]  → 초단위 (0, 600]

- 초 vs 분 겹침 시각들:
   00:01:01.017
   00:02:02.034
   00:03:03.051
   00:04:04.068
   00:05:05.085
   00:06:06.102
   00:07:07.119
   00:08:08.136
   00:09:09.153

- 초 vs 시 겹침 시각들:
   00:01:00.083
   00:02:00.167
   00:03:00.250
   00:04:00.334
   00:05:00.417
   00:06:00.501
   00:07:00.584
   00:08:00.668
   00:09:00.751

- 동시 겹침(초-분 & 초-시 동시에 같은 순간):
  (없음)

[카운트]
초-분: 9회
초-시: 9회
합계(중복 제거): 18회

[구간] (01:23:45, 02:00:00]  → 초단위 (5025, 7200]

- 초 vs 분 겹침 시각들:
   01:24:24.407
   01:25:25.424
   01:26:26.441
   01:27:27.458
   01:28:28.475
   01:29:29.492
   01:30:30.508
   01:31:31.525
   01:32:32.542
   01:33:33.559
  ... 총 36회

- 초 vs 시 겹침 시각들:
   01:24:07.010
   01:25:07.093
   01:26:07.177
   01:27:07.260
   01:28:07.344
   01:29:07.427
   01:30:07.510
   01:31:07.594
   01:32:07.677
   01:33:07.761
  ... 총 36회

- 동시 겹침(초-분 & 초-시 동시에 같은 순간):
  (없음)

[카운트]
초-분: 36회
초-시: 36회
합계(중복 제거): 72회

[구간] (01:05:05, 01:05:06]  → 초단위 (3905, 3906]

- 초 